# 야간 운전 시각 개선 시스템 — 모델 학습 (Colab)

**모델**: Zero-DCE + Retinex 기반 손실함수  
**학습**: `data/processed/` 통합 데이터셋 단일 학습

## 사전 조건
> **data_prep_colab.ipynb** 를 먼저 실행하여 `data/processed/` 를 준비하세요.  
> 또는 이 노트북 3번 셀에서 바로 데이터를 준비할 수 있습니다.

## 실행 순서
1. 환경 설치
2. Google Drive 마운트 및 저장소 로드
3. 데이터 확인 (없으면 자동 준비)
4. 모델 학습
5. 결과 시각화
6. 모델 Drive에 저장

---
## 1. 환경 설치

In [ ]:
# GPU 확인
import torch
print('CUDA 사용 가능:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')
else:
    print('⚠️  GPU 없음 — 런타임 > 런타임 유형 변경 > T4 GPU로 설정하세요.')

In [ ]:
# 필요 패키지 설치
%pip install -q gdown scikit-image tqdm onnx onnxruntime

---
## 2. Google Drive 마운트 및 저장소 로드

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Drive 내 경로 설정 (본인 경로에 맞게 수정)
DRIVE_CUSTOM_DATA = '/content/drive/MyDrive/night-vision-data'
DRIVE_MODEL_SAVE  = '/content/drive/MyDrive/night-vision-models'

import os
os.makedirs(DRIVE_MODEL_SAVE, exist_ok=True)
print('Drive 마운트 완료.')

In [ ]:
import os

REPO_URL    = 'https://github.com/mia2583/night-vision.git'
PROJECT_DIR = '/content/night-vision'

# ── 방법 A: GitHub 클론 ──────────────────────────
if not os.path.exists(PROJECT_DIR):
    !git clone {REPO_URL} {PROJECT_DIR}
else:
    !git -C {PROJECT_DIR} pull

# ── 방법 B: Drive에서 복사 (A 실패 시 주석 해제) ──
# import shutil
# DRIVE_PROJECT_ROOT = '/content/drive/MyDrive/night-vision'
# shutil.copytree(DRIVE_PROJECT_ROOT, PROJECT_DIR, dirs_exist_ok=True)

os.chdir(PROJECT_DIR)
print('작업 디렉터리:', os.getcwd())

---
## 3. 데이터 확인

`data_prep_colab.ipynb`를 실행한 경우 자동으로 인식됩니다.  
그렇지 않으면 아래 셀을 실행해 데이터를 준비합니다.

In [ ]:
import json
from pathlib import Path

DATA_DIR  = 'data/processed'
meta_path = Path(DATA_DIR) / 'metadata.json'

if meta_path.exists():
    with open(meta_path) as f:
        meta = json.load(f)
    print('데이터 준비 완료.')
    print(f"  학습: {meta['total_train']}쌍, 검증: {meta['total_val']}쌍, 테스트: {meta['total_test']}장")
else:
    print('⚠️  data/processed/ 없음 — 아래 셀에서 데이터를 준비하세요.')

In [ ]:
# data/processed/가 없을 때 여기서 직접 준비 (선택 실행)
import os, shutil

LOL_DIR    = 'data/lol'
CUSTOM_DIR = 'data/custom'
os.makedirs(LOL_DIR,    exist_ok=True)
os.makedirs(CUSTOM_DIR, exist_ok=True)

from utils.download_lol import download_lol_dataset
try:
    download_lol_dataset(LOL_DIR)
except RuntimeError as e:
    print(f'LOL 다운로드 실패: {e}')

if os.path.exists(DRIVE_CUSTOM_DATA):
    shutil.copytree(DRIVE_CUSTOM_DATA, CUSTOM_DIR, dirs_exist_ok=True)

from utils.prepare_data import prepare_data
meta = prepare_data(lol_dir=LOL_DIR, custom_dir=CUSTOM_DIR, output_dir=DATA_DIR)
print(f"학습: {meta['total_train']}쌍, 검증: {meta['total_val']}쌍")

---
## 4. 모델 학습

`data/processed/`의 통합 데이터셋으로 단일 학습합니다.

> ⚠️ **STEP 3, 4, 5 구현 후 활성화됩니다.**  
> `models/zerodce.py`, `models/losses.py`, `training/train.py` 필요

In [ ]:
# TODO: STEP 3, 4, 5 구현 후 아래 코드 활성화

# TRAIN_CONFIG = {
#     'data_dir':   DATA_DIR,
#     'epochs':     200,
#     'batch_size': 16,
#     'lr':         1e-4,
#     'input_size': 192,
#     'device':     'cuda',
#     'save_dir':   'models/pretrained',
#     'save_path':  'models/pretrained/zerodce_trained.pt',
# }
#
# from training.train import Trainer
# trainer = Trainer(**TRAIN_CONFIG)
# trainer.train()

print('[TODO] STEP 5 구현 후 활성화됩니다.')

---
## 5. 결과 시각화

> 학습 완료 후 실행하세요.

In [ ]:
# TODO: STEP 5 이후 활성화
# from utils.visualization import show_results
# show_results(model, test_ds, num_samples=5)

print('[TODO] STEP 5 구현 후 활성화됩니다.')

---
## 6. 모델 Google Drive에 저장

In [ ]:
import shutil, glob, os

model_files = glob.glob('models/pretrained/*.pt') + glob.glob('models/pretrained/*.onnx')

if model_files:
    for f in model_files:
        dst = os.path.join(DRIVE_MODEL_SAVE, os.path.basename(f))
        shutil.copy2(f, dst)
        print(f'저장됨: {dst}')
else:
    print('저장할 모델 파일이 없습니다. (학습 완료 후 실행하세요)')